# Liver-chip small-model action planner

Upload **small_model_colab.zip** from the AI4S workspace. In Colab, select a GPU runtime (your L4) and run the cells in order. The notebook downloads the Apache-licensed SmolLM2-135M-Instruct checkpoint, evaluates it, adapts it on 728 simulated policy examples and exports the selected model and results.

Labels come from our explicit heuristic policy, not expert or biological ground truth. The 364 test requests and six published compounds are excluded from training and checkpoint selection. Numerical explanations remain generated by the auditor. This notebook does not measure performance on your PC, Raspberry Pi or phone. Keep the exported ZIP before ending the runtime.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import zipfile
from google.colab import files

uploaded = files.upload()
package_name = "small_model_colab.zip"
if package_name not in uploaded:
    raise ValueError("Upload small_model_colab.zip from the prepared package.")
project_root = Path("/content/liver_chip_slm")
project_root.mkdir(exist_ok=True)
with zipfile.ZipFile(package_name) as package:
    for member in package.namelist():
        destination = (project_root / member).resolve()
        if not destination.is_relative_to(project_root.resolve()):
            raise ValueError("Unexpected path in the uploaded package.")
    package.extractall(project_root)
os.chdir(project_root)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "small_model/requirements.txt"], check=True)

## Check the runtime and frozen data

The baseline comparison is policy agreement. A high score does not establish better experimental decisions. Do not modify the policy or training prompts after inspecting model test results.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Enable the Colab GPU runtime before running this training experiment.")
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
subprocess.run([sys.executable, "-m", "small_model.check_planner"], check=True)
subprocess.run([sys.executable, "-m", "small_model.evaluate", "--partition", "test"], check=True)
manifest = json.loads(Path("small_model/data/manifest.json").read_text())
print({name: split["examples"] for name, split in manifest["splits"].items()})

## Evaluate the untuned checkpoint

Action choices use length-normalised log likelihood over the eight short codes. The raw choice is recorded separately from eligibility checks and rules fallback. Capture the resolved Hub revision and use the same checkpoint for training.

In [ ]:
base_model = "HuggingFaceTB/SmolLM2-135M-Instruct"
subprocess.run([sys.executable, "-m", "small_model.evaluate", "--model", base_model, "--name", "untuned", "--partition", "validation", "--device", "cuda"], check=True)
base_result = json.loads(Path("small_model/results/untuned_validation.json").read_text())
base_revision = base_result["summary"]["model_metadata"]["resolved_revision"]
if not base_revision:
    raise RuntimeError("The downloaded model revision was not recorded.")
subprocess.run([sys.executable, "-m", "small_model.evaluate", "--model", base_model, "--revision", base_revision, "--name", "untuned", "--partition", "test", "--device", "cuda"], check=True)
print("Frozen base revision:", base_revision)

## Adapt on training examples only

Three epochs, batch size 8 and learning rate 0.00005 are fixed in advance. Only assistant action-code tokens contribute to the loss. Select the checkpoint with lowest validation loss. An existing checkpoint is preserved: use a new output path to run a new training experiment.

In [ ]:
checkpoint = "small_model/checkpoints/smollm2-135m-policy"
subprocess.run([sys.executable, "-m", "small_model.train", "--base-model", base_model, "--revision", base_revision, "--device", "cuda", "--epochs", "3", "--batch-size", "8", "--learning-rate", "0.00005", "--output", checkpoint], check=True)

## Evaluate the selected model and a CPU smoke benchmark

The engineering gate uses the full test set. The 42 published requests are demonstrations, not independent biological validation. The CPU timing run uses the first 14 published requests with one warm-up; it measures Colab's CPU, not a named edge device. Download the model to run a full CPU benchmark on the PC.

In [ ]:
for partition in ["validation", "test", "published"]:
    subprocess.run([sys.executable, "-m", "small_model.evaluate", "--model", checkpoint, "--name", "adapted", "--partition", partition, "--device", "cuda"], check=True)
subprocess.run([sys.executable, "-m", "small_model.evaluate", "--model", checkpoint, "--name", "adapted_cpu_smoke", "--partition", "published", "--device", "cpu", "--threads", "4", "--limit", "14"], check=True)
test_summary = json.loads(Path("small_model/results/adapted_test.json").read_text())["summary"]
print("Raw model policy agreement:", test_summary["raw"]["agreement"])
print("Delivered agreement after checks:", test_summary["delivered"]["agreement"])
print("Rules agreement:", test_summary["rules"]["agreement"])
print("Fallback rate:", test_summary["fallback_rate"])
print("Engineering gate passed:", test_summary["engineering_gate_passed"])
print("Inspect per-class counts; rare actions have very few examples.")

## Export the model and evidence

Download this ZIP and return it to the AI4S workspace. It contains the selected weights, tokenizer, training record and every evaluation choice. It excludes the downloaded base-model cache. If the gate fails, keep the model labelled experimental and report its failures rather than claiming successful deployment.

In [ ]:
export_path = Path("/content/liver_chip_slm_result.zip")
with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for folder in [Path(checkpoint), Path("small_model/results")]:
        for path in sorted(folder.rglob("*")):
            if path.is_file():
                archive.write(path, path.as_posix())
    for filename in ["small_model/data/manifest.json", "small_model/protocol.md", "small_model/README.md", "small_model/SOURCE_MODEL_CARD.md", "small_model/LICENSE.base-model.txt", "NOTICE.md", "LICENSE"]:
        archive.write(filename, filename)
print("Selected checkpoint:", checkpoint)
print("Export size (MB):", round(export_path.stat().st_size / 1e6, 1))
files.download(str(export_path))